## Setup, Imports, and Local CUDA Environment

In [ ]:
!pip install -q datasets trl polars kagglehub peft transformers bitsandbytes

In [ ]:
import datasets
import trl
print("Successfully imported datasets version:", datasets.__version__)
print("Successfully imported trl version:", trl.__version__)

In [ ]:
import multiprocessing

multiprocessing.cpu_count()

In [ ]:
import os
import zipfile
from pathlib import Path

import polars as pl
import torch
import kagglehub
from datasets import Dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    TrainingArguments,
 )
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training, TaskType
from trl import SFTTrainer, SFTConfig

# --- Local workspace path resolution ---
REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "data").exists() and (REPO_ROOT.parent / "data").exists():
    REPO_ROOT = REPO_ROOT.parent

TRAIN_CSV_PATH = REPO_ROOT / "data" / "raw" / "train.csv"
OUTPUT_DIR = REPO_ROOT / "artifacts" / "adapter"
SUBMISSION_ZIP_PATH = REPO_ROOT / "artifacts" / "submission.zip"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# --- Local Torch CUDA environment ---
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True,max_split_size_mb:64")
CUDA_VISIBLE_DEVICES = os.getenv("CUDA_VISIBLE_DEVICES", "0")
os.environ["CUDA_VISIBLE_DEVICES"] = CUDA_VISIBLE_DEVICES

USE_CUDA = torch.cuda.is_available()
CUDA_DEVICE_COUNT = torch.cuda.device_count() if USE_CUDA else 0
TRAIN_DTYPE = torch.bfloat16 if USE_CUDA else torch.float32

print(f"Repo root: {REPO_ROOT}")
print(f"Train CSV: {TRAIN_CSV_PATH}")
print(f"Output dir: {OUTPUT_DIR}")
print(f"Torch version: {torch.__version__}")
print(f"CUDA available: {USE_CUDA}")
print(f"CUDA_VISIBLE_DEVICES: {os.environ.get('CUDA_VISIBLE_DEVICES')}")
if USE_CUDA:
    for i in range(CUDA_DEVICE_COUNT):
        print(f"GPU {i}: {torch.cuda.get_device_name(i)}")

# --- Hyperparameters ---
SUBSAMPLE_SIZE = 600
LORA_RANK = 32
MAX_SEQ_LEN = 1024
NUM_EPOCHS = 1
GRAD_ACCUM = 4
LR = 2e-4

## Data Loading & Formatting

In [ ]:
# Download model
MODEL_PATH = kagglehub.model_download("metric/nemotron-3-nano-30b-a3b-bf16/transformers/default")

# Load and subsample data
train_df = pl.read_csv(TRAIN_CSV_PATH)
print(f"Total training samples available: {len(train_df)}")
train_df = train_df.sample(n=min(SUBSAMPLE_SIZE, len(train_df)), seed=42)

# Convert to Hugging Face Dataset
hf_dataset = Dataset.from_pandas(train_df.to_pandas())

# Initialize tokenizer to build the text
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

def build_training_text(example):
    prompt = example["prompt"]
    answer = example["answer"]

    user_msg = prompt + "\nPut your final answer inside \\boxed{}."
    assistant_msg = f"\\boxed{{{answer}}}"

    try:
        messages = [
            {"role": "user", "content": user_msg},
            {"role": "assistant", "content": assistant_msg},
        ]
        text = tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=False
        )
    except Exception:
        text = (
            f"<|im_start|>user\n{user_msg}<|im_end|>\n"
            f"<|im_start|>assistant\n{assistant_msg}<|im_end|>"
        )
    return {"text": text}

In [ ]:
hf_dataset = hf_dataset.map(
    build_training_text,
    remove_columns=hf_dataset.column_names # <--- This deletes 'id', 'prompt', and 'answer'
)

print(f"Dataset formatted. Example:\n{hf_dataset[0]['text'][:500]}")

## Model Loading & LoRA Setup

In [ ]:
# Load Model (QLoRA-friendly path for stable CUDA training)
if USE_CUDA:
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.bfloat16,
        bnb_4bit_use_double_quant=True,
    )
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_PATH,
        trust_remote_code=True,
        quantization_config=bnb_config,
        device_map="auto",
    )
    model = prepare_model_for_kbit_training(model)
else:
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_PATH,
        trust_remote_code=True,
        torch_dtype=torch.float32,
        device_map="cpu",
    )

model.config.use_cache = False
print(f"Model loaded. Vocab size: {len(tokenizer)}")

# Setup LoRA
lora_config = LoraConfig(
    r=LORA_RANK,
    lora_alpha=16,
    target_modules="all-linear", # Targets all linear layers for better reasoning performance
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.CAUSAL_LM,
 )

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

## Training with SFTTrainer

In [ ]:
training_args = SFTConfig(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=GRAD_ACCUM,
    num_train_epochs=NUM_EPOCHS,
    learning_rate=LR,
    logging_steps=5,
    bf16=USE_CUDA,
    fp16=False,
    max_grad_norm=1.0,
    optim="adamw_torch",
    lr_scheduler_type="cosine",
    warmup_ratio=0.1,
    save_strategy="no",
    report_to="none",
    dataset_text_field="text",
    max_length=MAX_SEQ_LEN,
    packing=False,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
 )

trainer = SFTTrainer(
    model=model,
    train_dataset=hf_dataset,
    processing_class=tokenizer,
    args=training_args,
 )

print("Starting training...")
trainer.train()

## Save and Package Submission

In [ ]:
# Save adapter weights and config
trainer.model.save_pretrained(str(OUTPUT_DIR))
print(f"Adapter files saved to {OUTPUT_DIR}:")
for f in os.listdir(OUTPUT_DIR):
    size = os.path.getsize(os.path.join(OUTPUT_DIR, f))
    print(f"  {f} ({size/1024:.1f} KB)")

# Package into submission.zip (required by competition)
zip_path = str(SUBMISSION_ZIP_PATH)
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    for fname in os.listdir(OUTPUT_DIR):
        fpath = os.path.join(OUTPUT_DIR, fname)
        zf.write(fpath, fname)  # files at zip root

print(f"\nCreated {zip_path} ({os.path.getsize(zip_path)/1024/1024:.1f} MB)")

# Verify it contains adapter_config.json (required)
with zipfile.ZipFile(zip_path, 'r') as zf:
    names = zf.namelist()
    print(f"Contents: {names}")
    assert "adapter_config.json" in names, "Missing adapter_config.json!"
    print("✓ submission.zip looks correct and is ready to submit!")